# N20 Pure-State Entanglement Slope Vs System Size

This Colab/A100 notebook runs canonical GPU Markov dynamics, streams final-state covariance batches through an entanglement-contour observer, saves batch-sharded sample- and `y0`-averaged strip contours, and extracts the full-`x` log-chord entropy slope at cycle 50.

## Configurations And Domain-Wall Geometry

| Nx | Ny values | nshell | protocol | samples | cycles | init state | dtype | dw_truncation | alpha_1 / alpha_2 | fit window |
|---:|---|---:|---|---:|---:|---|---|---|---|---|
| 20 | 30, 40, 50, 60, 80, 100, 120 | 1 | perfect_correction | 100 | 50 | default random pure state | complex128 | True | 1 / 30 | Ay=8..Ny//2 |

For `Nx=20`, the class uses `half=Nx//2=10` and `w=floor(0.2*Nx)=4`, so `DW_loc=[6, 14]`. The topological `alpha_1` slab is `x=6..14` inclusive; the trivial `alpha_2` regions are `x=0..5` and `x=15..19`.

Covariance histories are not saved. Contour products are saved in Ay-batch shards, and the Markov driver chooses `batch_size=None` using its A100 40GB auto-batching helper.

## 1. Setup

In [ ]:

from __future__ import annotations

import json
from pathlib import Path
import sys
import time

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)

BUNDLE_NAME = 'colab_large_entanglement_scaling_N20'


def is_bundle_root(path: Path) -> bool:
    return (path / 'src' / 'classA_U1FGTN_gpu.py').exists() and (path / 'src' / 'strip_entropy_streaming_gpu.py').exists()


candidates = []
for base in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([base, base / BUNDLE_NAME])
if IN_COLAB:
    candidates.extend([
        Path('/content/drive/MyDrive') / BUNDLE_NAME,
        Path('/content/drive/MyDrive/class_A_fermionic_adaptive_circuit') / BUNDLE_NAME,
    ])

seen = set()
BUNDLE_ROOT = None
for candidate in candidates:
    candidate = candidate.resolve()
    if candidate in seen:
        continue
    seen.add(candidate)
    if is_bundle_root(candidate):
        BUNDLE_ROOT = candidate
        break
if BUNDLE_ROOT is None:
    raise FileNotFoundError(f'Could not locate {BUNDLE_NAME} with required src files')

SRC_DIR = BUNDLE_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'[setup] bundle root: {BUNDLE_ROOT}')


## 2. Imports

In [ ]:

import gc
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from tqdm.auto import tqdm
import torch

from classA_U1FGTN_gpu import classA_U1FGTN_gpu
from strip_entropy_streaming_gpu import (
    HELPER_VERSION,
    StripContourAccumulator,
    entropy_curve_rows_from_accumulator,
    fit_full_x_log_chord,
    log_sin_chord,
    rel_to_root,
    write_contour_batch_files,
    write_json_atomic,
)

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for this Colab GPU campaign.')

torch.set_grad_enabled(False)
plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 180,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print(f'[setup] helper={HELPER_VERSION}')
print(f'[setup] torch={torch.__version__}')
print(f'[setup] cuda device: {torch.cuda.get_device_name(0)}')
free_bytes, total_bytes = torch.cuda.mem_get_info(torch.device('cuda:0'))
print(f'[setup] cuda free/total GiB: {free_bytes/1024**3:.2f}/{total_bytes/1024**3:.2f}')


## 3. Campaign Configuration

In [ ]:

NX = 20
NY_VALUES = [30, 40, 50, 60, 80, 100, 120]
NSHELL = 1
SAMPLES = 100
CYCLES = 50
FIT_CYCLE = 50
FIT_AY_MIN = 8
ALPHA_1 = 1.0
ALPHA_2 = 30.0
DW_TRUNCATION = True
INIT_MODE = 'default'
DTYPE = 'complex128'
DEVICE = 'cuda:0'
BACKEND = 'local'
SEQUENCE = 'raster_y'
POSTSELECT = False
PERFECT_CORRECTION = True
N_A = 0.5
BATCH_SIZE = None
RESUME = True
SKIP_EXISTING = True
AY_BATCH_SIZE = 4
CYCLE_BATCH_SIZE = 1
AUTOTUNE_FOR_A100_40GB = True
AUTOTUNE_REPEAT = 2
MEMORY_SAFETY_FRACTION = 0.85
EIGH_MEMORY_MULTIPLIER = 8.0
SAMPLE_CHUNK_CANDIDATES = [1, 2, 4, 8, 16]
Y0_CHUNK_CANDIDATES = [1, 2, 4, 6, 8, 12, 16, 24, 32, 'ny']
TARGET_SLOPE_FULL_X = 1.0 / 3.0

GPU_DATA_ROOT = BUNDLE_ROOT / 'gpu_data'
CAMPAIGN_NAME = 'pure_state_entanglement_slope_vs_system_size'
OUTPUT_ROOT = GPU_DATA_ROOT / CAMPAIGN_NAME
RUNS_ROOT = OUTPUT_ROOT / 'runs'
FIG_ROOT = OUTPUT_ROOT / 'figures'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
FIG_ROOT.mkdir(parents=True, exist_ok=True)
CANONICAL_ENTRY_POINT = 'classA_U1FGTN_gpu.run_markov_circuit'

campaign_config = {
    'Nx': NX,
    'Ny_values': NY_VALUES,
    'nshell': NSHELL,
    'samples': SAMPLES,
    'cycles': CYCLES,
    'fit_cycle': FIT_CYCLE,
    'fit_Ay_min': FIT_AY_MIN,
    'alpha_1': ALPHA_1,
    'alpha_2': ALPHA_2,
    'dw_truncation': DW_TRUNCATION,
    'init_mode': INIT_MODE,
    'dtype': DTYPE,
    'sequence': SEQUENCE,
    'perfect_correction': PERFECT_CORRECTION,
    'postselect': POSTSELECT,
    'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
    'gpu_target': 'A100 40GB',
}
display(Markdown('### Campaign config'))
display(pd.DataFrame([campaign_config]))


## 4. Shared Helpers

In [ ]:

def show_saved_figure(path: Path, title: str | None = None, note: str | None = None) -> None:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if title:
        display(Markdown(f'**{title}**'))
    if note:
        display(Markdown(note))
    display(Image(filename=str(path)))


def save_dataframe_atomic_csv(df: pd.DataFrame, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    df.to_csv(tmp, index=False)
    tmp.replace(path)


def update_gpu_data_index(name: str, kind: str, manifest_rel: str) -> None:
    index_path = GPU_DATA_ROOT / 'index.json'
    if index_path.exists():
        payload = json.loads(index_path.read_text())
    else:
        payload = {'bundle_root': str(BUNDLE_ROOT), 'gpu_data_root': str(GPU_DATA_ROOT), 'campaigns': []}
    campaigns = payload.setdefault('campaigns', [])
    campaigns[:] = [item for item in campaigns if item.get('name') != name]
    campaigns.append({'name': name, 'kind': kind, 'manifest': manifest_rel})
    payload['bundle_root'] = str(BUNDLE_ROOT)
    payload['gpu_data_root'] = str(GPU_DATA_ROOT)
    write_json_atomic(index_path, payload)


def plot_log_chord_fit(entropy_curves: pd.DataFrame, fit_row: dict, *, fig_path: Path, title: str) -> None:
    fig, ax = plt.subplots(figsize=(7.2, 4.6), constrained_layout=True)
    sub = entropy_curves[(entropy_curves['cycle'] == int(fit_row['cycle'])) & (entropy_curves['Ay'] > 0)].sort_values('Ay')
    x_all = sub['log_sin_pi_Ay_over_Ny'].to_numpy(dtype=np.float64)
    y_all = sub['entropy_mean'].to_numpy(dtype=np.float64)
    yerr = sub['entropy_sem'].to_numpy(dtype=np.float64)
    x_band_min = float(fit_row['fit_x_min'])
    x_band_max = float(fit_row['fit_x_max'])
    x_line = np.linspace(float(np.min(x_all)), float(np.max(x_all)), 300)
    y_line = float(fit_row['slope']) * x_line + float(fit_row['intercept'])
    ax.axvspan(x_band_min, x_band_max, color='0.5', alpha=0.18, label=f"fit Ay={int(fit_row['Ay_fit_min'])}..{int(fit_row['Ay_fit_max'])}")
    ax.errorbar(x_all, y_all, yerr=yerr, marker='o', ms=3.5, lw=1.3, capsize=2, color='tab:blue', label='full_x S(Ay) data')
    ax.plot(x_line, y_line, color='black', ls='--', lw=1.5, label=f"fit m={float(fit_row['slope']):.4f}; r2={float(fit_row['r2']):.4f}")
    ax.set_title(title)
    ax.set_xlabel(r'$\log[\sin(\pi A_y/N_y)]$')
    ax.set_ylabel(r'full_x $S(A_y)$')
    ax.legend(frameon=True, fontsize=8)
    fig.savefig(fig_path)
    plt.close(fig)


## 5. Run One System Size

In [ ]:

def run_one_system_size(Ny: int) -> dict:
    Ny = int(Ny)
    run_id = f'N{NX}x{Ny}_nsh{NSHELL}_dwtrunc{int(DW_TRUNCATION)}_C{CYCLES}_S{SAMPLES}'
    run_root = RUNS_ROOT / run_id
    contour_root = run_root / 'contour_batches'
    fig_root = run_root / 'figures'
    run_root.mkdir(parents=True, exist_ok=True)
    contour_root.mkdir(parents=True, exist_ok=True)
    fig_root.mkdir(parents=True, exist_ok=True)
    summary_path = run_root / 'run_summary.json'
    fit_path = run_root / 'fit_rows.csv'
    entropy_path = run_root / 'entropy_curves.csv'

    if SKIP_EXISTING and summary_path.exists() and fit_path.exists():
        summary = json.loads(summary_path.read_text())
        if summary.get('status') == 'complete':
            print(f'[skip] {run_id}')
            return summary

    config = {
        **campaign_config,
        'Ny': Ny,
        'run_id': run_id,
        'Ay_values': list(range(Ny // 2 + 1)),
    }
    accumulator = StripContourAccumulator(
        nx=NX,
        ny=Ny,
        ay_values=range(Ny // 2 + 1),
        cycles=[FIT_CYCLE],
        config=config,
    )
    checkpoint_path = run_root / 'accumulator_checkpoint.npz'

    def cycle_observer(*, cycle, G, batch_index, batch_start, batch_count):
        if int(cycle) != FIT_CYCLE:
            return
        print(f'[observer] {run_id}: cycle={cycle} batch={batch_index} samples {batch_start}:{batch_start + batch_count}', flush=True)
        accumulator.update(
            cycle=cycle,
            G=G,
            eps=1e-12,
            autotune=AUTOTUNE_FOR_A100_40GB,
            sample_chunk_candidates=SAMPLE_CHUNK_CANDIDATES,
            y0_chunk_candidates=Y0_CHUNK_CANDIDATES,
            autotune_repeat=AUTOTUNE_REPEAT,
            memory_safety_fraction=MEMORY_SAFETY_FRACTION,
            eigh_memory_multiplier=EIGH_MEMORY_MULTIPLIER,
        )
        accumulator.save_checkpoint(checkpoint_path)

    print(f'[run] {run_id} output={run_root}')
    model = classA_U1FGTN_gpu(
        Nx=NX,
        Ny=Ny,
        DW=True,
        nshell=NSHELL,
        filling_frac=0.5,
        alpha_1=ALPHA_1,
        alpha_2=ALPHA_2,
        trial_orbitals='X',
        dw_truncation=DW_TRUNCATION,
        triv_region_local_mode=False,
        device=DEVICE,
        dtype=DTYPE,
        backend=BACKEND,
    )
    result = model.run_markov_circuit(
        G_history=False,
        progress=False,
        cycles=CYCLES,
        postselect=POSTSELECT,
        perfect_correction=PERFECT_CORRECTION,
        samples=SAMPLES,
        init_mode=INIT_MODE,
        save=False,
        n_a=N_A,
        sequence=SEQUENCE,
        batch_size=BATCH_SIZE,
        return_data=False,
        cycle_observer=cycle_observer,
    )

    batch_records = write_contour_batch_files(
        accumulator=accumulator,
        output_dir=contour_root,
        ay_batch_size=AY_BATCH_SIZE,
        cycle_batch_size=CYCLE_BATCH_SIZE,
        config=config,
    )
    entropy_curves = pd.DataFrame(entropy_curve_rows_from_accumulator(accumulator)).sort_values(['cycle', 'Ay']).reset_index(drop=True)
    save_dataframe_atomic_csv(entropy_curves, entropy_path)
    fit_row = fit_full_x_log_chord(
        entropy_curves,
        nx=NX,
        ny=Ny,
        nshell=NSHELL,
        cycle=FIT_CYCLE,
        fit_ay_min=FIT_AY_MIN,
        target_slope=TARGET_SLOPE_FULL_X,
    )
    fit_rows = pd.DataFrame([fit_row])
    save_dataframe_atomic_csv(fit_rows, fit_path)
    fig_path = fig_root / 'entropy_vs_log_chord_cycle50.png'
    plot_log_chord_fit(entropy_curves, fit_row, fig_path=fig_path, title=f'cycle {FIT_CYCLE}, N{NX}x{Ny}, nshell={NSHELL}')

    summary = {
        'status': 'complete',
        'run_id': run_id,
        'config': config,
        'run_root': str(run_root),
        'run_root_relative': rel_to_root(run_root, GPU_DATA_ROOT),
        'contour_batches_relative': [rel_to_root(record['filename'], GPU_DATA_ROOT) for record in batch_records],
        'entropy_curves_relative': rel_to_root(entropy_path, GPU_DATA_ROOT),
        'fit_rows_relative': rel_to_root(fit_path, GPU_DATA_ROOT),
        'figure_relative': rel_to_root(fig_path, GPU_DATA_ROOT),
        'checkpoint_relative': rel_to_root(checkpoint_path, GPU_DATA_ROOT),
        'markov_result': result,
        'batching': {
            'markov_batch_size_requested': BATCH_SIZE,
            'markov_batch_size_resolved': result.get('batch_size'),
            'markov_batch_size_mode': result.get('batch_size_mode'),
            'markov_batch_size_auto_info': result.get('batch_size_auto_info'),
            'contour_autotune_enabled': AUTOTUNE_FOR_A100_40GB,
            'sample_chunk_candidates': SAMPLE_CHUNK_CANDIDATES,
            'y0_chunk_candidates': Y0_CHUNK_CANDIDATES,
            'memory_safety_fraction': MEMORY_SAFETY_FRACTION,
            'eigh_memory_multiplier': EIGH_MEMORY_MULTIPLIER,
        },
        'fit': fit_row,
        'created_unix': time.time(),
    }
    write_json_atomic(summary_path, summary)
    gc.collect()
    torch.cuda.empty_cache()
    return summary


## 6. Execute Campaign

In [ ]:

all_results = []
for Ny in tqdm(NY_VALUES, desc='Ny systems', unit='system'):
    all_results.append(run_one_system_size(Ny))

fit_summary = pd.concat([pd.read_csv(GPU_DATA_ROOT / result['fit_rows_relative']) for result in all_results], ignore_index=True)
fit_summary = fit_summary.sort_values('Ny').reset_index(drop=True)
summary_csv = OUTPUT_ROOT / 'slope_vs_system_size_summary.csv'
save_dataframe_atomic_csv(fit_summary, summary_csv)
np.savez_compressed(
    OUTPUT_ROOT / 'slope_vs_system_size_summary.npz',
    Ny=fit_summary['Ny'].to_numpy(dtype=np.int64),
    slope=fit_summary['slope'].to_numpy(dtype=np.float64),
    slope_err=fit_summary['slope_err'].to_numpy(dtype=np.float64),
    r2=fit_summary['r2'].to_numpy(dtype=np.float64),
    config_json=np.asarray(json.dumps(campaign_config, sort_keys=True)),
)

display(Markdown('### Slope vs system size summary'))
display(fit_summary)


## 7. Campaign Figures And Manifest

In [ ]:

fig, ax = plt.subplots(figsize=(7.2, 4.6), constrained_layout=True)
ax.errorbar(fit_summary['Ny'], fit_summary['slope'], yerr=fit_summary['slope_err'], marker='o', capsize=3, lw=1.4)
ax.axhline(TARGET_SLOPE_FULL_X, color='black', ls='--', lw=1.0, label='target 1/3')
ax.set_xlabel('Ny')
ax.set_ylabel('full_x log-chord slope')
ax.set_title(f'N{NX} slope vs system size, cycle {FIT_CYCLE}')
ax.legend(frameon=True)
fig_path_slope = FIG_ROOT / 'slope_vs_Ny_cycle50.png'
fig.savefig(fig_path_slope)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7.2, 4.6), constrained_layout=True)
ax.errorbar(1.0 / fit_summary['Ny'], fit_summary['slope'], yerr=fit_summary['slope_err'], marker='o', capsize=3, lw=1.4)
ax.axhline(TARGET_SLOPE_FULL_X, color='black', ls='--', lw=1.0, label='target 1/3')
ax.set_xlabel('1 / Ny')
ax.set_ylabel('full_x log-chord slope')
ax.set_title(f'N{NX} slope vs inverse system size, cycle {FIT_CYCLE}')
ax.legend(frameon=True)
fig_path_inv = FIG_ROOT / 'slope_vs_inverse_Ny_cycle50.png'
fig.savefig(fig_path_inv)
plt.close(fig)

manifest = {
    'campaign_name': CAMPAIGN_NAME,
    'campaign_config': campaign_config,
    'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
    'helper_version': HELPER_VERSION,
    'bundle_root': str(BUNDLE_ROOT),
    'gpu_data_root': str(GPU_DATA_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'output_root_relative': rel_to_root(OUTPUT_ROOT, GPU_DATA_ROOT),
    'runs': all_results,
    'summary_csv_relative': rel_to_root(summary_csv, GPU_DATA_ROOT),
    'figures_relative': [rel_to_root(fig_path_slope, GPU_DATA_ROOT), rel_to_root(fig_path_inv, GPU_DATA_ROOT)],
    'batch_saving': {'Ay_batch_size': AY_BATCH_SIZE, 'cycle_batch_size': CYCLE_BATCH_SIZE},
}
manifest_path = OUTPUT_ROOT / 'campaign_manifest.json'
write_json_atomic(manifest_path, manifest)
update_gpu_data_index(CAMPAIGN_NAME, 'pure_state_entanglement_slope_scaling', rel_to_root(manifest_path, GPU_DATA_ROOT))
show_saved_figure(fig_path_slope, 'Slope vs Ny')
show_saved_figure(fig_path_inv, 'Slope vs 1/Ny')
print(f'[done] campaign manifest: {manifest_path}')


## 8. Disconnect Runtime

In [ ]:

if IN_COLAB:
    from google.colab import runtime  # type: ignore
    runtime.unassign()
else:
    print('[done] not running in Colab; runtime disconnect skipped')
